<a href="https://colab.research.google.com/github/codeSamuraii/ml-experiments/blob/main/DimensionReductionVideo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dimensionality Reduction on Video
This notebook explores applying standard dimensionality reduction algorithms to video data.

**Two experiments:**
1. **Time dimension reduction** — Collapse a sequence of frames into fewer frames (or a single frame) using PCA and t-SNE.
2. **Channel reduction** — Reduce RGB pixel values to a single channel using PCA and compare with standard luminance-based greyscale.

## Setup
Install dependencies and generate a sample video.

In [ ]:
!pip install -q numpy scikit-learn Pillow matplotlib

### Generate sample video
We use ffmpeg's `testsrc2` to quickly produce a short 720p test video.

If you want the rotating cube video, run `python tests/samples/generate.py` instead (requires numpy, Pillow, and ffmpeg).

In [ ]:
import subprocess
import os

SAMPLE_DIR = "tests/samples"
os.makedirs(SAMPLE_DIR, exist_ok=True)
SAMPLE_VIDEO = os.path.join(SAMPLE_DIR, "720p.mp4")

if not os.path.exists(SAMPLE_VIDEO):
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "lavfi",
        "-i", "testsrc2=size=1280x720:rate=30:duration=10",
        "-c:v", "libx264", "-preset", "fast", "-crf", "18",
        "-pix_fmt", "yuv420p",
        SAMPLE_VIDEO,
    ], capture_output=True, check=True)
    print(f"Generated {SAMPLE_VIDEO}")
else:
    print(f"Using existing {SAMPLE_VIDEO}")

## Loading video frames
Extract frames from the video using ffmpeg and load them as numpy arrays.

In [ ]:
import numpy as np
from PIL import Image
import io
import struct

def extract_frames(video_path, max_frames=120, resize=(320, 180)):
    """Extract frames from a video file using ffmpeg.

    Returns an array of shape (N, H, W, 3) with uint8 RGB values.
    """
    w, h = resize
    cmd = [
        "ffmpeg", "-i", video_path,
        "-vframes", str(max_frames),
        "-s", f"{w}x{h}",
        "-f", "rawvideo", "-pix_fmt", "rgb24",
        "pipe:1",
    ]
    result = subprocess.run(cmd, capture_output=True, check=True)
    raw = result.stdout
    frame_size = w * h * 3
    n_frames = len(raw) // frame_size
    frames = np.frombuffer(raw[:n_frames * frame_size], dtype=np.uint8)
    frames = frames.reshape(n_frames, h, w, 3)
    return frames

frames = extract_frames(SAMPLE_VIDEO, max_frames=120, resize=(320, 180))
print(f"Loaded {frames.shape[0]} frames of shape {frames.shape[1:]} (H, W, C)")

### Sample frames from the video

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
indices = np.linspace(0, len(frames) - 1, 6, dtype=int)
for ax, idx in zip(axes, indices):
    ax.imshow(frames[idx])
    ax.set_title(f"Frame {idx}")
    ax.axis("off")
plt.suptitle("Sample frames from input video", fontsize=14)
plt.tight_layout()
plt.show()

## Experiment 1: Time dimension reduction
We treat the video as a 3D array of shape `(N_frames, H*W, C)`.

To reduce the time dimension, we flatten each frame into a vector and apply dimensionality reduction across the frame (time) axis.

- **PCA** finds orthogonal directions of maximum variance across frames.
- **t-SNE** is a nonlinear method that preserves local structure.

We reduce to `k` components and reshape back to images.

In [ ]:
from sklearn.decomposition import PCA

def pca_reduce_time(frames, n_components=1):
    """Reduce time dimension of video frames using PCA.

    Parameters
    ----------
    frames : ndarray of shape (N, H, W, 3)
        Input video frames as uint8 RGB.
    n_components : int
        Number of output frames (principal components).

    Returns
    -------
    reduced : ndarray of shape (n_components, H, W, 3)
        Reduced frames, rescaled to 0-255 uint8.
    pca : PCA
        Fitted PCA object.
    """
    n, h, w, c = frames.shape
    # Each frame becomes a row; columns are pixel values
    flat = frames.reshape(n, -1).astype(np.float64)

    pca = PCA(n_components=n_components)
    # components_ has shape (n_components, n_pixels*3)
    pca.fit(flat)
    components = pca.components_

    # Rescale each component to 0-255
    reduced = []
    for comp in components:
        img = comp.reshape(h, w, c)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
        reduced.append(img.astype(np.uint8))

    return np.array(reduced), pca

# Reduce to 1, 3, and 5 components
for k in [1, 3, 5]:
    reduced, pca = pca_reduce_time(frames, n_components=k)
    var_explained = pca.explained_variance_ratio_.sum() * 100
    print(f"PCA k={k}: variance explained = {var_explained:.1f}%")

    fig, axes = plt.subplots(1, k, figsize=(5 * k, 4))
    if k == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.imshow(reduced[i])
        ax.set_title(f"Component {i+1} ({pca.explained_variance_ratio_[i]*100:.1f}%)")
        ax.axis("off")
    plt.suptitle(f"PCA time reduction (k={k}, {var_explained:.1f}% variance)", fontsize=13)
    plt.tight_layout()
    plt.show()

### t-SNE time reduction
t-SNE embeds high-dimensional data into a lower-dimensional space. Unlike PCA, it is non-linear and focuses on preserving local neighbourhood structure.

Since t-SNE produces an embedding (coordinates) rather than reconstructed images, we use a different approach: we compute the t-SNE embedding of all frames in 2D, then select representative frames (e.g. closest to the centroid or at the extremes of the embedding).

In [ ]:
from sklearn.manifold import TSNE

def tsne_reduce_time(frames, n_select=5, perplexity=30, random_state=42):
    """Select representative frames using t-SNE embedding.

    Parameters
    ----------
    frames : ndarray of shape (N, H, W, 3)
    n_select : int
        Number of representative frames to return.
    perplexity : float
        t-SNE perplexity (should be less than N).
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    selected_frames : ndarray of shape (n_select, H, W, 3)
    selected_indices : ndarray of int
    embedding : ndarray of shape (N, 2)
    """
    n = frames.shape[0]
    flat = frames.reshape(n, -1).astype(np.float64)

    effective_perplexity = min(perplexity, n - 1)
    tsne = TSNE(n_components=2, perplexity=effective_perplexity,
                random_state=random_state)
    embedding = tsne.fit_transform(flat)

    # Select frames spread across the embedding using k-means on the 2D coords
    from sklearn.cluster import KMeans
    kmeans = KMeans(n_clusters=n_select, random_state=random_state, n_init=10)
    kmeans.fit(embedding)

    # Pick the frame closest to each cluster center
    selected = []
    for center in kmeans.cluster_centers_:
        dists = np.linalg.norm(embedding - center, axis=1)
        selected.append(np.argmin(dists))
    selected = np.array(sorted(set(selected)))

    return frames[selected], selected, embedding

selected_frames, selected_idx, embedding = tsne_reduce_time(frames, n_select=5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot embedding
ax1.scatter(embedding[:, 0], embedding[:, 1], c=np.arange(len(frames)),
            cmap="viridis", s=10, alpha=0.7)
ax1.scatter(embedding[selected_idx, 0], embedding[selected_idx, 1],
            c="red", s=100, marker="x", linewidths=2, zorder=5)
ax1.set_title("t-SNE embedding of frames (colour = time)")
ax1.set_xlabel("t-SNE 1")
ax1.set_ylabel("t-SNE 2")
cbar = plt.colorbar(ax1.collections[0], ax=ax1)
cbar.set_label("Frame index")

# Show selected frames
ax2.axis("off")
ax2.set_title(f"Representative frames (n={len(selected_idx)})")

fig2, axes = plt.subplots(1, len(selected_idx), figsize=(4 * len(selected_idx), 4))
if len(selected_idx) == 1:
    axes = [axes]
for ax, idx in zip(axes, selected_idx):
    ax.imshow(frames[idx])
    ax.set_title(f"Frame {idx}")
    ax.axis("off")
plt.suptitle("t-SNE selected representative frames", fontsize=13)
plt.tight_layout()
plt.show()

## Experiment 2: RGB channel reduction (PCA greyscale)
Standard greyscale uses the luminance formula: `Y = 0.2126 R + 0.7152 G + 0.0722 B`.

Here we use PCA to find the single direction of maximum variance across the RGB channels, projecting each pixel's 3-value colour down to 1 value. This is a data-driven "greyscale" — let's see if it looks similar to standard luminance greyscale.

In [ ]:
def pca_greyscale(image):
    """Reduce RGB to 1 channel using PCA on pixel colour values.

    Parameters
    ----------
    image : ndarray of shape (H, W, 3), uint8

    Returns
    -------
    grey : ndarray of shape (H, W), uint8
    pca : PCA
        Fitted PCA showing the learned projection.
    """
    h, w, c = image.shape
    pixels = image.reshape(-1, 3).astype(np.float64)

    pca = PCA(n_components=1)
    projected = pca.fit_transform(pixels).ravel()

    # Rescale to 0-255
    projected = (projected - projected.min()) / (projected.max() - projected.min() + 1e-8) * 255
    return projected.reshape(h, w).astype(np.uint8), pca

def luminance_greyscale(image):
    """Standard luminance-based greyscale conversion."""
    return (0.2126 * image[:, :, 0] +
            0.7152 * image[:, :, 1] +
            0.0722 * image[:, :, 2]).astype(np.uint8)

# Compare on several frames
sample_indices = np.linspace(0, len(frames) - 1, 4, dtype=int)

fig, axes = plt.subplots(4, 3, figsize=(14, 16))
for row, idx in enumerate(sample_indices):
    frame = frames[idx]

    pca_grey, pca_obj = pca_greyscale(frame)
    lum_grey = luminance_greyscale(frame)

    axes[row, 0].imshow(frame)
    axes[row, 0].set_title(f"Frame {idx} — Original RGB")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(lum_grey, cmap="gray")
    axes[row, 1].set_title("Luminance greyscale")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(pca_grey, cmap="gray")
    weights = pca_obj.components_[0]
    axes[row, 2].set_title(
        f"PCA greyscale\n"
        f"weights: R={weights[0]:.3f}, G={weights[1]:.3f}, B={weights[2]:.3f}"
    )
    axes[row, 2].axis("off")

plt.suptitle("RGB → 1 channel: Luminance vs PCA", fontsize=15)
plt.tight_layout()
plt.show()

## Analysis

### Time reduction
- **PCA component 1** captures the most common visual pattern across all frames — essentially a "temporal average" weighted by variance. For a video with a moving object, this tends to produce a blurred composite of all positions.
- **Higher PCA components** capture differences between frames — motion, lighting changes, etc.
- **t-SNE** groups visually similar frames together and helps select a diverse, representative subset.

### Channel reduction
- The **PCA greyscale** weights are data-driven and will vary per image/video. For natural images, PCA typically finds weights close to the luminance formula because green contributes most variance in typical scenes.
- For synthetic videos with unusual colour distributions, PCA may diverge noticeably from standard greyscale.

### Further experiments
You could extend this by trying:
- **UMAP** for time reduction (faster than t-SNE for large frame counts)
- **Kernel PCA** with different kernels for non-linear time reduction
- **ICA (Independent Component Analysis)** to separate independent temporal signals
- **NMF (Non-negative Matrix Factorization)** which guarantees non-negative components